# Chunking For Retrieval

A retrieval-augmented generation (RAG) system cannot retrieve an entire knowledge base intelligently until it has defined the units that may be retrieved. This notebook turns policy documents into chunks and makes the most important design choice visible: a chunk must be small enough to match a question and large enough to keep the evidence needed for an answer.

## Learning Objectives

By the end of this notebook, you should be able to:

- distinguish source documents, chunks, and retrieved context,
- implement fixed-size chunking with overlap and metadata,
- compare chunk configurations with concrete evidence checks,
- choose a chunking strategy for the handbook corpus.

## Where Chunking Sits In RAG

```mermaid
graph LR
    A["Source documents"] --> B["Split into chunks"]
    B --> C["Embed chunks"]
    C --> D["Index vectors"]
    Q["Question"] --> E["Retrieve top-k chunks"]
    D --> E
    E --> F["Grounded answer"]
```

RAG first retrieves evidence and then asks a language model to answer
using that evidence. Chunking is upstream of every later decision: if a
useful sentence is broken away from its condition or never retrieved,
generation cannot repair the missing evidence reliably.

<div align="center">
  <img src="assets/01-rag-architecture.png" alt="Retrieval-augmented generation architecture" width="700" style="max-width: 100%; height: auto;" />
  <p><em>Retrieval-augmented generation architecture. Source: <a href="https://commons.wikimedia.org/wiki/File:RAG_schema.svg" target="_blank" rel="noopener noreferrer">Wikimedia Commons</a></em></p>
</div>


## A Note On Tokens And Boundaries

Production chunkers often measure length in tokens because embedding and
generation models consume tokens rather than characters. We keep the first
splitter word-based so its behavior is inspectable, then map that reasoning
to tokenizer-aware implementations such as the Hugging Face Tokenizers library.

This excerpt from the Hugging Face course makes the bridge concrete: byte-pair
encoding applies learned merge rules, so a visible word may become several
subword tokens. A chunk budget based on characters or whitespace is therefore
only an approximation of the eventual model input budget.

<div align="center">
  <img src="assets/01-bpe-tokenization.png" alt="Byte pair encoding tokenization example" width="900" style="max-width: 100%; height: auto;" />
  <p><em>Byte pair encoding tokenization example. Source: <a href="https://huggingface.co/docs/course/chapter6/5" target="_blank" rel="noopener noreferrer">Hugging Face Course</a> (official documentation excerpt)</em></p>
</div>


## Load The Shared Handbook Corpus

The JSON file is intentionally shared across the notebook path because later lessons index, retrieve, generate from, and evaluate the same source documents.


In [ ]:
import json
from pathlib import Path

# All notebooks use this compact handbook so retrieval choices are comparable.
corpus_path = Path("data") / "course_handbook.json"
documents = json.loads(corpus_path.read_text())

print(f"Loaded {len(documents)} source documents.")
for document in documents[:3]:
    print(f"- {document['doc_id']}: {document['title']}")

A **document** is an authored source item. A **chunk** is an indexing unit
derived from it. Metadata such as `doc_id`, `title`, and `section` must
travel with each chunk so a retrieved answer can cite its source.


In [ ]:
sample = documents[0]
print(sample["title"])
print(sample["text"])
print("Words:", len(sample["text"].split()))

## Fixed-Size Windows With Overlap

A sliding window takes `chunk_size` words at a time. The next window moves
forward by `chunk_size - overlap` words. Overlap repeats boundary text so
a phrase that crosses a split may still appear together in one retrievable
chunk. The cost is duplicated indexing and potentially repetitive context.


In [ ]:
def sliding_windows(words: list[str], chunk_size: int, overlap: int) -> list[list[str]]:
    """Split words into overlapping windows while rejecting unsafe settings."""

    if chunk_size <= 0 or not 0 <= overlap < chunk_size:
        raise ValueError("Use chunk_size > 0 and 0 <= overlap < chunk_size.")
    # Overlap reduces the step so neighboring windows repeat boundary words.
    step = chunk_size - overlap
    return [words[start : start + chunk_size] for start in range(0, len(words), step)]

The splitter below preserves provenance. `chunk_id` is stable within one configuration; it is not a universal source identifier.


In [ ]:
def chunk_documents(
    source_documents: list[dict[str, str]], chunk_size: int, overlap: int
) -> list[dict[str, object]]:
    """Create retrieval units with the original document metadata attached."""

    chunks: list[dict[str, object]] = []
    # Preserve source fields on each chunk so retrieval can later cite evidence.
    for document in source_documents:
        windows = sliding_windows(document["text"].split(), chunk_size, overlap)
        for position, words in enumerate(windows):
            chunks.append(
                {
                    "chunk_id": f"{document['doc_id']}-{position}",
                    "doc_id": document["doc_id"],
                    "title": document["title"],
                    "section": document["section"],
                    "text": " ".join(words),
                }
            )
    return chunks

Run a compact configuration first. Notice that one source document can become several retrievable records.


In [ ]:
chunks = chunk_documents(documents, chunk_size=18, overlap=5)
print(f"{len(documents)} documents became {len(chunks)} chunks.")
for chunk in chunks[:3]:
    print(chunk["chunk_id"], "=>", chunk["text"])

## Compare Chunking Choices

Bigger chunks carry more context but may mix unrelated details. Smaller
chunks can match narrow questions precisely but risk losing qualifying
conditions. Here we compare corpus size and duplication, then check whether
important evidence phrases remain intact.


In [ ]:
configurations = [
    {"name": "short_no_overlap", "chunk_size": 10, "overlap": 0},
    {"name": "short_overlap", "chunk_size": 10, "overlap": 3},
    {"name": "balanced", "chunk_size": 18, "overlap": 5},
    {"name": "long", "chunk_size": 32, "overlap": 5},
]

for config in configurations:
    candidate = chunk_documents(
        documents, int(config["chunk_size"]), int(config["overlap"])
    )
    # Repeated overlap increases words stored and scored by the retriever.
    indexed_words = sum(len(str(item["text"]).split()) for item in candidate)
    source_words = sum(len(item["text"].split()) for item in documents)
    print(
        config["name"],
        "chunks=",
        len(candidate),
        "duplication=",
        round(indexed_words / source_words, 2),
    )

The following checks are deliberately simple, but they expose a genuine
retrieval risk: a result about refunds is less useful if it retrieves
`within 30 days` separately from `duplicate payments`.


In [ ]:
# These phrases stand in for evidence an answer must retrieve intact.
evidence_phrases = {
    "refunds": "both transactions refer",
    "mfa-device": "authentication can be",
    "api-rate-limit": "per API key.",
    "privacy": "identity documents in",
}


def phrase_is_retrievable(
    candidate_chunks: list[dict[str, object]], doc_id: str, phrase: str
) -> bool:
    """Return whether any one chunk keeps a required evidence phrase intact."""

    # A phrase split across two chunks cannot ground one retrieved passage.
    return any(
        item["doc_id"] == doc_id and phrase.lower() in str(item["text"]).lower()
        for item in candidate_chunks
    )

In [ ]:
for config in configurations:
    candidate = chunk_documents(
        documents, int(config["chunk_size"]), int(config["overlap"])
    )
    preserved = sum(
        phrase_is_retrievable(candidate, doc_id, phrase)
        for doc_id, phrase in evidence_phrases.items()
    )
    print(
        f"{config['name']}: {preserved}/{len(evidence_phrases)} required phrases preserved"
    )

For the ten-word no-overlap configuration, each inspected phrase crosses
a window boundary and is therefore unavailable as one evidence unit.
Overlap restores those passages in this example. A phrase-preservation
check does **not** prove retrieval quality: it does not ask whether a query
ranks the chunk highly. It is an upstream sanity check. Notebook 05 will
evaluate rankings with labeled questions.


## Paragraph Boundaries Versus Windows

Natural boundaries such as headings or paragraphs are often better than
arbitrary windows, especially for long authored pages. Our small corpus has
one compact paragraph per policy, so each source document itself is also a
useful baseline chunking strategy.


In [ ]:
def whole_document_chunks(
    source_documents: list[dict[str, str]],
) -> list[dict[str, str]]:
    """Use one compact authored policy as one retrieval unit."""

    return [
        {
            "chunk_id": f"{document['doc_id']}-whole",
            "doc_id": document["doc_id"],
            "title": document["title"],
            "section": document["section"],
            "text": document["text"],
        }
        for document in source_documents
    ]


baseline_chunks = whole_document_chunks(documents)
print(f"Whole-document baseline: {len(baseline_chunks)} chunks")

For this short corpus, the whole-document baseline is strong: each policy
describes one topic and is already below a reasonable context budget. For a
long handbook chapter, we would need boundaries or windows inside a source
document. Good chunking is data-dependent rather than a universal number.


## Retrieval Units Are An Information Contract

Chunking is not simply pre-processing. It declares which evidence can be
returned together. Consider three failure patterns:

| Choice | Benefit | Retrieval risk |
| --- | --- | --- |
| Very short chunks | precise keyword match | condition and answer split apart |
| Very long chunks | complete local context | several topics compete in one vector |
| Heavy overlap | boundary evidence retained | duplicate chunks consume top-k positions |

This is why chunking must be evaluated against questions: a text splitter
is acceptable only when it makes relevant evidence retrievable within the
context budget of the downstream answer prompt.


## Measure Repetition In Retrieved Candidates

Overlap can preserve evidence while making adjacent chunks nearly
identical. A simple diagnostic counts how often neighboring chunks from the
same document share words. It does not replace ranking evaluation, but it
makes excessive overlap visible before vectors are even built.


In [ ]:
def neighboring_overlap(candidate_chunks: list[dict[str, object]]) -> list[float]:
    """Measure repeated vocabulary between adjacent chunks of one source."""

    scores = []
    for left, right in zip(candidate_chunks, candidate_chunks[1:]):
        if left["doc_id"] != right["doc_id"]:
            continue
        # Jaccard overlap gives a small, interpretable repetition diagnostic.
        left_words = set(str(left["text"]).lower().split())
        right_words = set(str(right["text"]).lower().split())
        scores.append(
            len(left_words & right_words) / max(len(left_words | right_words), 1)
        )
    return scores

In [ ]:
for config in configurations:
    candidate = chunk_documents(
        documents, int(config["chunk_size"]), int(config["overlap"])
    )
    repetition = neighboring_overlap(candidate)
    mean_overlap = sum(repetition) / len(repetition) if repetition else 0.0
    print(f"{config['name']}: mean neighboring vocabulary overlap={mean_overlap:.2f}")

A configuration with high neighboring repetition may cause the retriever to
return two versions of the same evidence when the application needed two
different policies. Later evaluation should therefore inspect both
relevance and diversity in the retrieved context.


## Exercise - Tune A Boundary-Safe Splitter

Use `chunk_documents` to select a window configuration that:

1. preserves all four `evidence_phrases`;
2. keeps each chunk at no more than 18 words;
3. creates fewer than 30 chunks;
4. has duplication below `1.40`.

Print the winning settings and explain in one sentence why it is a
defensible alternative to whole-document chunks for longer sources.


## Correction


In [ ]:
# Search candidate settings against explicit quality and storage constraints.
viable = []
source_words = sum(len(item["text"].split()) for item in documents)
for size in range(8, 31, 2):
    for overlap in range(0, min(size, 7)):
        candidate = chunk_documents(documents, size, overlap)
        preserved = all(
            phrase_is_retrievable(candidate, doc_id, phrase)
            for doc_id, phrase in evidence_phrases.items()
        )
        duplication = (
            sum(len(str(item["text"]).split()) for item in candidate) / source_words
        )
        if preserved and size <= 18 and len(candidate) < 30 and duplication < 1.40:
            viable.append((size, overlap, len(candidate), duplication))

# Prefer fewer retrieval units, then less duplicate indexed content.
chosen = sorted(viable, key=lambda result: (result[2], result[3]))[0]
print("chunk_size, overlap, chunks, duplication =", chosen)

**Reference answer:** The selected configuration respects the 18-word
retrieval budget while retaining the inspected evidence and keeping
duplication bounded. Unlike one-policy-per-chunk, a windowed splitter can
be applied when future source pages grow longer than the desired retrieval
context. The precise winner is less important than showing the checks
behind the decision.


## Pitfalls And Next Step

- A chunker that looks neat may separate an answer from its qualifier.
- Overlap increases storage and can crowd repeated passages into top-k
  results.
- Token-aware splitting matters when real embedding models impose input
  limits.

Next, we will convert chunks to vectors and ask whether a question actually
retrieves the relevant source.

## Further Reading

- Hugging Face: [Tokenizers](https://huggingface.co/docs/tokenizers/python/latest/index.html)
- LangChain: [Docs](https://docs.langchain.com/oss/python/langchain/overview) / [Text Splitters](https://docs.langchain.com/oss/python/integrations/splitters)
- LlamaIndex: [Docs](https://developers.llamaindex.ai/python/framework/) / [Node Parsing](https://docs.llamaindex.ai/en/stable/module_guides/loading/node_parsers/)
